In [ ]:
!pip install mlforecast scikit-learn xgboost catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.1/128.1 kB 1.4 MB/s eta 0:00:00 MB/s eta 0:00:0101
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 33.2 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.2/97.2 MB 38.5 MB/s eta 0:00:00m eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 MB 23.4 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 52.3 MB/s eta 0:00:0031m54.9 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.9/263.9 kB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 53.5 MB/s eta 0:00:0031m72.4 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.6/806.6 kB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 613.5/613.5 kB 59.3 MB/s eta 0:0

In [ ]:
import time
import numpy as np
import pandas as pd
import json

from sklearn.metrics import mean_absolute_error, mean_squared_error

# === CarbonTracker ===
import io
import re
from contextlib import redirect_stdout
from carbontracker.tracker import CarbonTracker as Tracker

# === Eco2AI ===
import eco2ai
eco2ai_tracker = eco2ai.Tracker(project_name="eco2ai", file_name="eco2ai_emissions.csv", ignore_warnings=True)

In [ ]:
from mlforecast import MLForecast
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

print("Tudo ok!")

Tudo ok!


In [ ]:
class CarbonTracker:
  def __init__(self, epochs=1, verbose=1):
    self.epochs = epochs
    self.verbose = verbose
    self.buffer = io.StringIO()
    self.tracker = None
    self.co2_g = None
    self.energy_kwh = None

  def __enter__(self):
    self._stdout_ctx = redirect_stdout(self.buffer)
    self._stdout_ctx.__enter__()
    self.tracker = Tracker(epochs=self.epochs, verbose=self.verbose)
    self.tracker.epoch_start()
    return self

  def __exit__(self, exc_type, exc, tb):
    self.tracker.epoch_end()
    self.tracker.stop()
    self._stdout_ctx.__exit__(exc_type, exc, tb)
    output = self.buffer.getvalue()

    match_co2 = re.search(r"CO2eq:\s*([\d\.]+)\s*g", output)
    self.co2_g = float(match_co2.group(1)) if match_co2 else None

    match_energy = re.search(r"Energy:\s*([\d\.]+)\s*kWh", output)
    self.energy_kwh = float(match_energy.group(1)) if match_energy else None

In [ ]:
def load_series(path, unique_id="series_1"):
    df = pd.read_csv(path, index_col='date')
    df = df.reset_index().rename(columns={'date': 'ds', 'value': 'y'})
    df['unique_id'] = unique_id
    df["ds"] = pd.to_datetime(df["ds"])    # Precisei adicionar pra ML

    return df

def smape(y_true, y_pred, decimals: int = 2) -> float:
    numerator = np.abs(y_true - y_pred)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    return round(float(np.mean(numerator / (denominator + 1e-10)) * 100), decimals)

def run_forecast(train_df, true_df, freq, season_length, dataset_name):
  results = []

  models = [
        ("RandomForest", RandomForestRegressor(n_estimators=100)),
        ("XGBoost", XGBRegressor(n_estimators=100)),
        ("CatBoost", CatBoostRegressor(verbose=0)),
  ]

  if freq == "h":
    lags = [1, 2, 3, 6, 12, 24, 48]
  elif freq == "d":
    lags = [1, 2, 3, 7, 14]

  for model_name, model in models:

    print("Rodando Modelo: ", model_name, "pro Dataset: ", dataset_name)

    fcst = MLForecast(
        models={model_name: model},
        freq=freq,
        lags=lags
    )

    eco2ai_tracker.start()

    with CarbonTracker() as carbontracker:
      start = time.time()

      fcst.fit(train_df)
      forecast = fcst.predict(h=len(true_df))

      end = time.time()

    eco2ai_tracker.stop()

    y_true = true_df['y'].values
    y_pred = forecast[model_name].values

    carbontracker_g = carbontracker.co2_g
    carbontracker_kWh = carbontracker.energy_kwh
    carbontracker_Wh = carbontracker_kWh * 1000 if carbontracker_kWh else None

    eco2ai_data = pd.read_csv("eco2ai_emissions.csv").iloc[-1]
    eco2ai_kWh = eco2ai_data["power_consumption(kWh)"]
    eco2ai_Wh = eco2ai_kWh * 1000
    eco2ai_kg = eco2ai_data["CO2_emissions(kg)"]
    eco2ai_g = eco2ai_kg * 1000

    results.append({
      "Dataset": dataset_name,
      "Modelo": model_name,
      "Parâmetros": str(model),
      "SMAPE": smape(y_true, y_pred),
      "MAE": mean_absolute_error(y_true, y_pred),
      "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
      "CarbonTracker CO₂ (g)": carbontracker_g,
      "CarbonTracker Consumo (Wh)": carbontracker_Wh,
      "Eco2AI CO₂ (g)": eco2ai_g,
      "Eco2AI Consumo (Wh)": eco2ai_Wh,
      "Tempo (s)": end - start,
      "y_true": json.dumps(y_true.tolist()),
      "y_pred": json.dumps(y_pred.tolist())
    })

  return results

In [ ]:
data_path = "/home/liaan/Documentos/Benchmark ZB/data"

In [ ]:
etth2 = load_series(f'{data_path}/ETTh2.csv', 'etth2')
electricity = load_series(f'{data_path}/Electricity.csv', 'electricity')
traffic = load_series(f'{data_path}/Transporte.csv', 'traffic')
covid19 = load_series(f'{data_path}/Covid-19.csv', 'covid19')
# mortality = load_series('data/Mortalidade.csv', 'mortality')
retail = load_series(f'{data_path}/Varejo.csv', 'retail')
wike2000 = load_series(f'{data_path}/Wike2000.csv', 'wike2000')
carbon = load_series(f'{data_path}/Carbon.csv', 'carbon')

In [ ]:
etth2_train = etth2[(etth2['ds'] >= '2016-07-12 06:00:00') & (etth2['ds'] <= '2016-09-20 05:00:00')]
etth2_true = etth2[etth2['ds'] > '2016-09-20 05:00:00'].iloc[:24]

electricity_train = electricity[(electricity['ds'] >= '2013-06-28 07:00:01') & (electricity['ds'] <= '2013-09-06 06:00:01')]
electricity_true = electricity[electricity['ds'] > '2013-09-06 06:00:01'].iloc[:24]

traffic_train = traffic[(traffic['ds'] >= '2018-04-06 14:00:00') & (traffic['ds'] <= '2018-06-15 13:00:00')]
traffic_true = traffic[traffic['ds'] > '2018-06-15 13:00:00'].iloc[:24]

covid19_train = covid19[(covid19['ds'] >= '2021-03-11') & (covid19['ds'] <= '2022-03-09')]
covid19_true = covid19[covid19['ds'] > '2022-03-09'].iloc[:7]

#mortality_train = mortality[(mortality['ds'] >= '2021-02-23') & (mortality['ds'] <= '2022-02-21')]
#mortality_true = mortality[mortality['ds'] > '2022-02-21'].iloc[:7]

retail_train = retail[(retail['ds'] >= '2017-01-01') & (retail['ds'] <= '2017-12-30')]
retail_true = retail[retail['ds'] > '2017-12-30'].iloc[:7]

wike2000_train = wike2000[(wike2000['ds'] >= '2012-07-08') & (wike2000['ds'] <= '2013-07-06')]
wike2000_true = wike2000[wike2000['ds'] > '2013-07-06'].iloc[:7]

carbon_train = carbon[(carbon['ds'] >= '2025-02-14') & (carbon['ds'] <= '2026-02-12')]
carbon_true = carbon[carbon['ds'] > '2026-02-12'].iloc[:7]

In [ ]:
all_results = []

all_results += run_forecast(etth2_train, etth2_true, freq='h', season_length=24, dataset_name='ETTh2')
all_results += run_forecast(electricity_train, electricity_true, freq='h', season_length=24, dataset_name='Electricity')
all_results += run_forecast(traffic_train, traffic_true, freq='h', season_length=24, dataset_name='Transporte')
all_results += run_forecast(covid19_train, covid19_true, freq='d', season_length=7, dataset_name='Covid-19')
#all_results += run_forecast(mortality_train, mortality_true, freq='d', season_length=7, dataset_name='Mortalidade')
all_results += run_forecast(retail_train, retail_true, freq='d', season_length=7, dataset_name='Varejo')
all_results += run_forecast(wike2000_train, wike2000_true, freq='d', season_length=7, dataset_name='Wike2000')
all_results += run_forecast(carbon_train, carbon_true, freq='d', season_length=7, dataset_name='Carbon')

Rodando Modelo:  RandomForest pro Dataset:  ETTh2


/home/liaan/Documentos/Benchmark ZB/Neurips 2026/zbvenv/lib/python3.11/site-packages/eco2ai/emission_track.py:515: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'N/A' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  attributes_dataframe.loc[row_index] = attributes_array
/home/liaan/Documentos/Benchmark ZB/Neurips 2026/zbvenv/lib/python3.11/site-packages/eco2ai/emission_track.py:515: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.9408905506134033' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  attributes_dataframe.loc[row_index] = attributes_array
/home/liaan/Documentos/Benchmark ZB/Neurips 2026/zbvenv/lib/python3.11/site-packages/eco2ai/emission_track.py:515: FutureWarning: Setting an item of incompatible dtype is deprecated and will rais

Rodando Modelo:  XGBoost pro Dataset:  ETTh2


/home/liaan/Documentos/Benchmark ZB/Neurips 2026/zbvenv/lib/python3.11/site-packages/eco2ai/emission_track.py:515: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'N/A' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  attributes_dataframe.loc[row_index] = attributes_array
/home/liaan/Documentos/Benchmark ZB/Neurips 2026/zbvenv/lib/python3.11/site-packages/eco2ai/emission_track.py:515: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.8754749298095703' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  attributes_dataframe.loc[row_index] = attributes_array
/home/liaan/Documentos/Benchmark ZB/Neurips 2026/zbvenv/lib/python3.11/site-packages/eco2ai/emission_track.py:515: FutureWarning: Setting an item of incompatible dtype is deprecated and will rais

Rodando Modelo:  CatBoost pro Dataset:  ETTh2


...

/home/liaan/Documentos/Benchmark ZB/Neurips 2026/zbvenv/lib/python3.11/site-packages/eco2ai/emission_track.py:515: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'N/A' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  attributes_dataframe.loc[row_index] = attributes_array
/home/liaan/Documentos/Benchmark ZB/Neurips 2026/zbvenv/lib/python3.11/site-packages/eco2ai/emission_track.py:515: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.6077156066894531' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  attributes_dataframe.loc[row_index] = attributes_array
/home/liaan/Documentos/Benchmark ZB/Neurips 2026/zbvenv/lib/python3.11/site-packages/eco2ai/emission_track.py:515: FutureWarning: Setting an item of incompatible dtype is deprecated and will rais

Rodando Modelo:  XGBoost pro Dataset:  Carbon


/home/liaan/Documentos/Benchmark ZB/Neurips 2026/zbvenv/lib/python3.11/site-packages/eco2ai/emission_track.py:515: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'N/A' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  attributes_dataframe.loc[row_index] = attributes_array
/home/liaan/Documentos/Benchmark ZB/Neurips 2026/zbvenv/lib/python3.11/site-packages/eco2ai/emission_track.py:515: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.5739943981170654' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  attributes_dataframe.loc[row_index] = attributes_array
/home/liaan/Documentos/Benchmark ZB/Neurips 2026/zbvenv/lib/python3.11/site-packages/eco2ai/emission_track.py:515: FutureWarning: Setting an item of incompatible dtype is deprecated and will rais

Rodando Modelo:  CatBoost pro Dataset:  Carbon


/home/liaan/Documentos/Benchmark ZB/Neurips 2026/zbvenv/lib/python3.11/site-packages/eco2ai/emission_track.py:515: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'N/A' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  attributes_dataframe.loc[row_index] = attributes_array
/home/liaan/Documentos/Benchmark ZB/Neurips 2026/zbvenv/lib/python3.11/site-packages/eco2ai/emission_track.py:515: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.8388969898223877' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  attributes_dataframe.loc[row_index] = attributes_array
/home/liaan/Documentos/Benchmark ZB/Neurips 2026/zbvenv/lib/python3.11/site-packages/eco2ai/emission_track.py:515: FutureWarning: Setting an item of incompatible dtype is deprecated and will rais

In [ ]:
results_df = pd.DataFrame(all_results)
results_df.to_csv("experimentos_ml.csv", index=False)
results_df

,Dataset,Modelo,Parâmetros,SMAPE,MAE,RMSE,CarbonTracker CO₂ (g),CarbonTracker Consumo (Wh),Eco2AI CO₂ (g),Eco2AI Consumo (Wh),Tempo (s),y_true,y_pred
0,ETTh2,RandomForest,RandomForestRegressor(),20.96,6.771467,8.500905,0.001633,0.015823,0.002674,0.023653,0.448601,"[25.35650062561035, 25.136999130249023, 27.333...","[25.732299938201905, 25.72571994781494, 25.732..."
1,ETTh2,XGBoost,"XGBRegressor(base_score=None, booster=None, ca...",19.36,6.170565,8.037044,0.001573,0.015238,0.002532,0.022393,0.376004,"[25.35650062561035, 25.136999130249023, 27.333...","[25.038986206054688, 24.706754684448242, 24.89..."
2,ETTh2,CatBoost,"CatBoostRegressor(loss_function='RMSE', verbos...",15.96,5.232525,7.354678,0.002598,0.025170,0.003299,0.029180,0.584575,"[25.35650062561035, 25.136999130249023, 27.333...","[26.690334400514736, 27.273893988399145, 27.71..."
3,Electricity,RandomForest,RandomForestRegressor(),9.93,0.073954,0.149355,0.002471,0.023938,0.003204,0.028337,0.547240,"[-1.376244106473734, -1.2267854542201622, -0.8...","[-1.403984393904747, -1.209235962154592, -0.18..."
4,Electricity,XGBoost,"XGBRegressor(base_score=None, booster=None, ca...",21.05,0.180516,0.269628,0.000668,0.006472,0.001998,0.017670,0.152892,"[-1.376244106473734, -1.2267854542201622, -0.8...","[-1.401699185371399, -1.2174557447433472, 0.07..."
5,Electricity,CatBoost,"CatBoostRegressor(loss_function='RMSE', verbos...",10.35,0.105156,0.131261,0.002230,0.021602,0.003017,0.026684,0.496268,"[-1.376244106473734, -1.2267854542201622, -0.8...","[-1.3526986956722638, -1.188026044321824, -0.5..."
6,Transporte,RandomForest,RandomForestRegressor(),13.27,76.507500,132.846114,0.002079,0.020139,0.002930,0.025909,0.472237,"[680.0, 810.0, 1131.0, 1346.0, 1194.0, 792.0, ...","[693.35, 733.62, 1016.1, 1374.54, 1131.99, 831..."
7,Transporte,XGBoost,"XGBRegressor(base_score=None, booster=None, ca...",38.54,76.619410,138.746706,0.000504,0.004881,0.001818,0.016080,0.112118,"[680.0, 810.0, 1131.0, 1346.0, 1194.0, 792.0, ...","[642.0361938476562, 725.8562622070312, 1014.60..."
8,Transporte,CatBoost,"CatBoostRegressor(loss_function='RMSE', verbos...",38.45,79.228495,137.304024,0.002253,0.021826,0.003056,0.027024,0.521380,"[680.0, 810.0, 1131.0, 1346.0, 1194.0, 792.0, ...","[658.7804124714141, 731.8366368922091, 1031.42..."
9,Covid-19,RandomForest,RandomForestRegressor(),48.80,2176.412857,2662.503682,0.000515,0.004988,0.001791,0.015843,0.114476,"[4184, 4194, 3614, 3116, 2503, 2568, 2876]","[4338.7, 4479.43, 4883.33, 5458.16, 5674.89, 6..."
